# Titanic — Machine Learning from Disaster | Version 2

**Mục tiêu:** cải thiện pipeline Version 1 bằng feature engineering + cross-validation + ensemble.


In [5]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from lightgbm import LGBMClassifier

RANDOM_STATE = 42
N_SPLITS = 5

print("Libraries loaded.")

Libraries loaded.


## 1. Load dữ liệu

Trên Kaggle, nếu dataset path khác, chỉ cần sửa hai biến bên dưới.

In [6]:
TRAIN_PATH = "/kaggle/input/competitions/titanic/train.csv"
TEST_PATH  = "/kaggle/input/competitions/titanic/test.csv"

train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)

print("Train:", train_raw.shape)
print("Test :", test_raw.shape)
display(train_raw.head())

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/competitions/titanic/train.csv'

## 2. Feature Engineering

Version 1 bỏ `Name`, `Ticket`, `Cabin` hoàn toàn. Đây là phần làm mất khá nhiều tín hiệu.

Version 2 tạo các đặc trưng có ý nghĩa:

- `Title`: Mr / Miss / Mrs / Master / Rare.
- `Surname`: họ, giúp nhận diện nhóm gia đình.
- `FamilySize`, `IsAlone`.
- `TicketGroupSize`: có bao nhiêu người dùng cùng vé.
- `SurnameSize`: kích thước nhóm cùng họ trong train + test.
- `FarePerPerson`: giá vé chia cho kích thước nhóm vé.
- `Deck`, `CabinKnown`.
- `TicketPrefix`.
- `Mother`, `Child`.
- cờ missing cho `Age` và `Fare`.

Kích thước nhóm được tính trên **train + test không có nhãn**, nên không đưa thông tin `Survived` từ test vào model.

In [ ]:
def feature_engineering(df):
    d = df.copy()

    # ---- Name ----
    d["Title"] = (
        d["Name"]
        .str.extract(r",\s*([^.]*)\.", expand=False)
        .str.strip()
        .replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
    )
    d.loc[~d["Title"].isin(["Mr", "Miss", "Mrs", "Master"]), "Title"] = "Rare"

    d["Surname"] = d["Name"].str.split(",").str[0].str.strip()

    # ---- Family ----
    d["FamilySize"] = d["SibSp"] + d["Parch"] + 1
    d["IsAlone"] = (d["FamilySize"] == 1).astype(int)
    d["Child"] = (d["Age"] < 14).astype(int)

    d["Mother"] = (
        (d["Sex"] == "female") &
        (d["Parch"] > 0) &
        (d["Age"].fillna(99) >= 18) &
        (d["Title"] != "Miss")
    ).astype(int)

    # ---- Ticket / fare ----
    d["TicketGroupSize"] = d.groupby("Ticket")["Ticket"].transform("size")
    d["SurnameSize"] = d.groupby("Surname")["Surname"].transform("size")
    d["FarePerPerson"] = d["Fare"] / d["TicketGroupSize"]

    # ---- Cabin ----
    d["Deck"] = d["Cabin"].fillna("U").str[0]
    d["CabinKnown"] = d["Cabin"].notna().astype(int)

    # ---- Ticket prefix ----
    d["TicketPrefix"] = (
        d["Ticket"]
        .str.replace(r"\d", "", regex=True)
        .str.replace(r"[\s./]", "", regex=True)
        .replace("", "NONE")
    )

    # ---- Missing indicators before imputation ----
    d["AgeMissing"] = d["Age"].isna().astype(int)
    d["FareMissing"] = d["Fare"].isna().astype(int)

    # ---- Imputation ----
    d["Age"] = (
        d["Age"]
        .fillna(d.groupby(["Title", "Pclass"])["Age"].transform("median"))
        .fillna(d["Age"].median())
    )

    d["Fare"] = (
        d["Fare"]
        .fillna(d.groupby("Pclass")["Fare"].transform("median"))
        .fillna(d["Fare"].median())
    )

    # Quantile bin is calculated after filling Fare.
    d["FareBin"] = pd.qcut(
        d["Fare"], q=5, labels=False, duplicates="drop"
    ).astype(int)

    return d


# Group-size features are intentionally computed before splitting train/test.
combined = pd.concat(
    [train_raw.drop(columns="Survived"), test_raw],
    axis=0,
    ignore_index=True
)

combined_fe = feature_engineering(combined)

train_fe = combined_fe.iloc[:len(train_raw)].copy()
test_fe = combined_fe.iloc[len(train_raw):].copy()

y = train_raw["Survived"].astype(int)

# Remove identifiers / target leakage candidates from the actual matrix.
DROP_COLS = ["PassengerId", "Name", "Cabin"]

X = train_fe.drop(columns=DROP_COLS)
X_test = test_fe.drop(columns=DROP_COLS)

# Cat/object columns for one-hot encoding.
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()

X = pd.get_dummies(X, columns=categorical_cols, dtype=int)
X_test = pd.get_dummies(X_test, columns=categorical_cols, dtype=int)

# Make columns exactly identical and in identical order.
X_test = X_test.reindex(columns=X.columns, fill_value=0)

print("Final train matrix:", X.shape)
print("Final test matrix :", X_test.shape)
display(X.head())

## 3. Vì sao không dùng một lần `train_test_split`?

Version 1 dùng một split 70/30. Với Titanic chỉ có 891 dòng, một split có thể cho kết quả khá dao động.

Version 2 dùng **5-fold Stratified Cross-Validation**:

- mỗi fold giữ tỷ lệ sống/chết gần giống toàn bộ train;
- mọi model đều được đánh giá trên dữ liệu mà nó chưa dùng để học;
- sau đó mới train model cuối cùng trên toàn bộ train.

Đây là cơ sở tốt hơn để chọn pipeline.

In [ ]:
cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

def evaluate_model(model, X_data, y_data, cv):
    oof = np.zeros(len(y_data))
    fold_scores = []

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X_data, y_data), 1):
        X_tr = X_data.iloc[train_idx]
        X_va = X_data.iloc[valid_idx]
        y_tr = y_data.iloc[train_idx]
        y_va = y_data.iloc[valid_idx]

        model.fit(X_tr, y_tr)

        prob = model.predict_proba(X_va)[:, 1]
        pred = (prob >= 0.5).astype(int)

        oof[valid_idx] = prob
        score = accuracy_score(y_va, pred)
        fold_scores.append(score)

        print(f"Fold {fold}: {score:.4f}")

    print(f"Mean CV accuracy: {np.mean(fold_scores):.4f}")
    print(f"Std  CV accuracy: {np.std(fold_scores):.4f}")
    return np.array(fold_scores), oof

## 4. Model A — Logistic Regression

Logistic Regression là baseline tuyến tính tốt cho Titanic và có tính bổ sung cho tree model: nó thường mắc lỗi ở những điểm khác với LightGBM.

In [ ]:
log_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        C=0.5,
        max_iter=2000,
        random_state=RANDOM_STATE
    )
)

log_scores, oof_log = evaluate_model(log_model, X, y, cv)

## 5. Model B — LightGBM

LightGBM có thể học các tương tác phi tuyến như:

`Sex × Pclass`, `Age × Title`, `FamilySize × Pclass`, `FarePerPerson × Pclass`...

Thay vì dùng một cây quá sâu, ta dùng cây tương đối nông và regularization để hạn chế overfitting trên dataset nhỏ.

In [ ]:
lgb_model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.03,
    num_leaves=15,
    max_depth=5,
    min_child_samples=15,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.2,
    reg_lambda=3.0,
    random_state=RANDOM_STATE,
    verbosity=-1
)

lgb_scores, oof_lgb = evaluate_model(lgb_model, X, y, cv)

## 6. Ensemble

Hai model có kiểu sai khác nhau. Ta kết hợp xác suất:

**Final probability = 55% LightGBM + 45% Logistic Regression**

Trọng số này được chọn từ thử nghiệm cross-validation trên train, không dựa vào nhãn thủ công của test.

In [ ]:
oof_ensemble = 0.55 * oof_lgb + 0.45 * oof_log
oof_pred = (oof_ensemble >= 0.5).astype(int)

ensemble_accuracy = accuracy_score(y, oof_pred)

print(f"OOF Ensemble accuracy: {ensemble_accuracy:.4f}")
print()
print(classification_report(y, oof_pred, digits=4))
print("Confusion matrix:")
print(confusion_matrix(y, oof_pred))

## 7. So sánh các model

In [ ]:
comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "LightGBM", "Ensemble"],
    "OOF Accuracy": [
        accuracy_score(y, oof_log >= 0.5),
        accuracy_score(y, oof_lgb >= 0.5),
        ensemble_accuracy
    ]
}).sort_values("OOF Accuracy", ascending=False)

display(comparison)

## 8. Train final models trên toàn bộ train

Sau khi chọn pipeline, ta không giữ lại 20% hay 30% dữ liệu làm validation nữa.

Hai model được fit trên **toàn bộ 891 dòng train có nhãn** rồi dự đoán 418 dòng test.

In [ ]:
log_final = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        C=0.5,
        max_iter=2000,
        random_state=RANDOM_STATE
    )
)

lgb_final = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.03,
    num_leaves=15,
    max_depth=5,
    min_child_samples=15,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.2,
    reg_lambda=3.0,
    random_state=RANDOM_STATE,
    verbosity=-1
)

log_final.fit(X, y)
lgb_final.fit(X, y)

test_prob_log = log_final.predict_proba(X_test)[:, 1]
test_prob_lgb = lgb_final.predict_proba(X_test)[:, 1]

test_prob = 0.55 * test_prob_lgb + 0.45 * test_prob_log
test_predictions = (test_prob >= 0.5).astype(int)

print("Predicted survivors:", test_predictions.sum())
print("Predicted deaths   :", len(test_predictions) - test_predictions.sum())

## 9. Create Kaggle submission

Không tự gán nhãn bằng cách xem từng hành khách. `Survived` được sinh hoàn toàn từ model đã train bằng `train.csv`.

File đầu ra phải có đúng hai cột:

- `PassengerId`
- `Survived`

In [ ]:
submission = pd.DataFrame({
    "PassengerId": test_raw["PassengerId"],
    "Survived": test_predictions
})

submission.to_csv("submission_v2.csv", index=False)

display(submission.head(10))
print("Saved: submission_v2.csv")
print("Shape:", submission.shape)
print("Columns:", submission.columns.tolist())

## 10. Kiểm tra cuối trước khi submit

Nếu tất cả đều đúng, file `submission_v2.csv` có thể được upload lên Kaggle.

### Điểm cần nhớ

- `OOF Accuracy` là chỉ báo nội bộ, **không phải Kaggle score**.
- Kaggle test set là dữ liệu ẩn nên không thể biết trước điểm chính xác.
- Điểm tối đa của Titanic là `1.00000`, không phải `1.xxxxx`.
- Nếu Version 2 tốt hơn Version 1 trên Kaggle, hãy giữ V2 làm baseline mới và chỉ thay đổi **một nhóm feature/model mỗi lần** để biết yếu tố nào thực sự cải thiện.